In [ ]:
import glob
import os
import pandas as pd
import json
from zipfile import ZipFile
os.getcwd()
os.chdir("C:\\Users\\c.coles\\Documents\\Python\\BGG DE Project")

In [ ]:
input_folder_path = ".\\BGG-Pipeline\\Data\\zip\\"
output_folder_path = ".\\BGG-Pipeline\\Data\\csv\\"

In [ ]:
list_of_files = glob.glob(f"{input_folder_path}*.zip") # * means all if need specific format then *.csv
latest_file_path = max(list_of_files, key=os.path.getmtime)
latest_file_name = os.path.basename(latest_file_path)

In [ ]:
with ZipFile(f"{input_folder_path}{latest_file_name}", 'r') as zObject:
  zObject.extract("boardgames_ranks.csv", path=f"{output_folder_path}")
zObject.close()

In [ ]:
import duckdb
con = duckdb.connect(database=':memory:')

In [ ]:
con.execute("""
    CREATE TABLE IF NOT EXISTS BGG_FILE_EXTRACT (
        id INTEGER PRIMARY KEY,
        filename VARCHAR,
        filedate DATE,
        content JSON
    )
""")

In [ ]:
bgg_ranks = pd.read_csv(f"{output_folder_path}boardgames_ranks.csv")
json_string = bgg_ranks.to_json(orient='records', indent=2)
json_string

In [ ]:
con.execute("""
  INSERT INTO BGG_FILE_EXTRACT values (?, ?, ? ,?)
""", [2,"A", "2020-03-19" , json_string])


In [ ]:
result = con.execute("""
    select * from BGG_FILE_EXTRACT
""").fetchall()
result
